# Project 4 — Panorama Stitcher
**ESIN, UIR — Introduction to Computer Vision — Spring 2026**  
**Groupe_Alaoui** : Mohamed Anouar FARESS · Mohammed Yassine BEN MEZIANE · Imad Eddine EL KHORASSANI · Anys LOUZAL

---
### Pipeline overview
1. Load overlapping images
2. Detect keypoints (Harris from scratch)
3. Extract descriptors (local patch)
4. Match keypoints (SSD / NCC)
5. Estimate homography (cv2.findHomography + RANSAC)
6. Warp images to a common coordinate system
7. Blend overlapping regions (alpha blending)
8. Display & save result

## 0. Imports & configuration

In [1]:
from __future__ import annotations

import numpy as np
import cv2
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
from pathlib import Path
from scipy.ndimage import gaussian_filter

# Display settings
plt.rcParams['figure.figsize'] = (14, 6)
plt.rcParams['image.cmap'] = 'gray'

DATA_DIR = Path('../data/images')
OUTPUT_DIR = Path('../data/output')
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

print('Libraries loaded successfully.')

Libraries loaded successfully.


## 1. Load images

In [2]:
def load_image(path: str | Path) -> np.ndarray:
    """
    Load an image from disk in BGR (OpenCV default) and convert to RGB.

    Parameters
    ----------
    path : str or Path
        Path to the image file.

    Returns
    -------
    np.ndarray
        RGB image as uint8 array of shape (H, W, 3).
    """
    img = cv2.imread(str(path))
    if img is None:
        raise FileNotFoundError(f'Image not found: {path}')
    return cv2.cvtColor(img, cv2.COLOR_BGR2RGB)


# --- Load your image set here ---
# Example: three overlapping shots of a scene
img_paths = sorted(DATA_DIR.glob('set1/*.jpeg'))
images = [load_image(p) for p in img_paths]

print(f'Loaded {len(images)} images')
fig, axes = plt.subplots(1, len(images))
for ax, img, p in zip(axes, images, img_paths):
    ax.imshow(img)
    ax.set_title(p.name)
    ax.axis('off')
plt.suptitle('Input images')
plt.tight_layout()
plt.show()

Loaded 3 images


/var/folders/0h/51gcfhf96z5cb9mhps40vflh0000gn/T/ipykernel_23803/3996832937.py:34: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


## 2. Keypoint detection — Harris corner detector (from scratch)
> **Course concept**: corner detection (Lab 4). We implement the full Harris response from the structure tensor.

In [3]:
def harris_corners(
    img: np.ndarray,
    k: float = 0.04,
    sigma: float = 1.0,
    threshold_ratio: float = 0.01,
    nms_radius: int = 5,
) -> np.ndarray:
    """
    Detect Harris corners in a grayscale image.

    Steps:
      1. Compute image gradients Ix, Iy using Sobel.
      2. Build the structure tensor M = [[Ix2, IxIy],[IxIy, Iy2]] with Gaussian weighting.
      3. Compute Harris response R = det(M) - k * trace(M)^2.
      4. Threshold and apply non-maximum suppression.

    Parameters
    ----------
    img : np.ndarray
        Grayscale image (H, W), float.
    k : float
        Harris sensitivity parameter (typically 0.04–0.06).
    sigma : float
        Gaussian window size for structure tensor smoothing.
    threshold_ratio : float
        Fraction of max(R) used as detection threshold.
    nms_radius : int
        Radius for non-maximum suppression.

    Returns
    -------
    np.ndarray
        Array of shape (N, 2) containing (row, col) of detected corners.
    """
    gray = img.astype(np.float32)

    # 1. Gradients
    Ix = cv2.Sobel(gray, cv2.CV_64F, 1, 0, ksize=3)
    Iy = cv2.Sobel(gray, cv2.CV_64F, 0, 1, ksize=3)

    # 2. Structure tensor elements (Gaussian smoothing)
    Ix2  = gaussian_filter(Ix * Ix, sigma=sigma)
    Iy2  = gaussian_filter(Iy * Iy, sigma=sigma)
    IxIy = gaussian_filter(Ix * Iy, sigma=sigma)

    # 3. Harris response
    det   = Ix2 * Iy2 - IxIy ** 2
    trace = Ix2 + Iy2
    R = det - k * (trace ** 2)

    # 4. Threshold
    threshold = threshold_ratio * R.max()
    R_thresh = np.where(R > threshold, R, 0)

    # 5. Non-maximum suppression (local max in nms_radius window)
    from scipy.ndimage import maximum_filter
    local_max = maximum_filter(R_thresh, size=2 * nms_radius + 1)
    corners = np.argwhere((R_thresh == local_max) & (R_thresh > 0))

    return corners  # shape (N, 2): rows, cols


def visualize_keypoints(img: np.ndarray, corners: np.ndarray, title: str = '') -> None:
    """Overlay detected corners on the image."""
    fig, ax = plt.subplots()
    ax.imshow(img)
    ax.scatter(corners[:, 1], corners[:, 0], s=10, c='lime', linewidths=0.5, marker='+')
    ax.set_title(title)
    ax.axis('off')
    plt.show()


# --- Test on the first two images ---
gray0 = cv2.cvtColor(images[0], cv2.COLOR_RGB2GRAY)
gray1 = cv2.cvtColor(images[1], cv2.COLOR_RGB2GRAY)

corners0 = harris_corners(gray0, threshold_ratio=0.01)
corners1 = harris_corners(gray1, threshold_ratio=0.01)

print(f'Image 0: {len(corners0)} corners detected')
print(f'Image 1: {len(corners1)} corners detected')

visualize_keypoints(images[0], corners0, title='Harris corners — image 0')
visualize_keypoints(images[1], corners1, title='Harris corners — image 1')

Image 0: 6696 corners detected
Image 1: 14805 corners detected


/var/folders/0h/51gcfhf96z5cb9mhps40vflh0000gn/T/ipykernel_23803/162064309.py:70: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


## 3. Descriptor extraction
> Extract a small local patch around each keypoint as a simple descriptor.

In [4]:
def extract_descriptors(
    img_gray: np.ndarray,
    corners: np.ndarray,
    patch_size: int = 16,
) -> tuple[np.ndarray, np.ndarray]:
    """
    Extract normalized flat patch descriptors around each keypoint.

    Parameters
    ----------
    img_gray : np.ndarray
        Grayscale image (H, W).
    corners : np.ndarray
        Corner positions (N, 2) in (row, col) format.
    patch_size : int
        Side length of the patch (must be even).

    Returns
    -------
    valid_corners : np.ndarray
        Corners that are far enough from image borders, shape (M, 2).
    descriptors : np.ndarray
        Normalized descriptors, shape (M, patch_size*patch_size).
    """
    half = patch_size // 2
    H, W = img_gray.shape
    descriptors = []
    valid = []

    for r, c in corners:
        # Skip keypoints too close to borders
        if r - half < 0 or r + half >= H or c - half < 0 or c + half >= W:
            continue
        patch = img_gray[r - half:r + half, c - half:c + half].astype(np.float32)
        # Normalize: zero mean, unit std
        patch = (patch - patch.mean()) / (patch.std() + 1e-8)
        descriptors.append(patch.flatten())
        valid.append((r, c))

    return np.array(valid), np.array(descriptors)


valid0, desc0 = extract_descriptors(gray0, corners0)
valid1, desc1 = extract_descriptors(gray1, corners1)

print(f'Descriptors — image 0: {desc0.shape}')
print(f'Descriptors — image 1: {desc1.shape}')

Descriptors — image 0: (6652, 256)
Descriptors — image 1: (14758, 256)


## 4. Keypoint matching (SSD)
> **Course concept**: Sum of Squared Differences (SSD) for template matching.

In [5]:
def match_descriptors_ssd(
    desc1: np.ndarray,
    desc2: np.ndarray,
    ratio_threshold: float = 0.75,
) -> np.ndarray:
    """
    Match descriptors using SSD with Lowe's ratio test.

    Parameters
    ----------
    desc1, desc2 : np.ndarray
        Descriptor matrices, shape (N1, D) and (N2, D).
    ratio_threshold : float
        Lowe's ratio test threshold (default 0.75).

    Returns
    -------
    np.ndarray
        Array of shape (M, 2) with matched indices (idx1, idx2).
    """
    matches = []
    for i, d in enumerate(desc1):
        diffs = np.sum((desc2 - d) ** 2, axis=1)  # SSD to all desc2
        sorted_idx = np.argsort(diffs)
        best, second = diffs[sorted_idx[0]], diffs[sorted_idx[1]]
        if best < ratio_threshold ** 2 * second:   # ratio test
            matches.append((i, sorted_idx[0]))
    return np.array(matches)


def visualize_matches(
    img1: np.ndarray,
    kp1: np.ndarray,
    img2: np.ndarray,
    kp2: np.ndarray,
    matches: np.ndarray,
    max_display: int = 50,
    title: str = 'Matches',
) -> None:
    """Draw lines between matched keypoints on side-by-side images."""
    H1, W1 = img1.shape[:2]
    canvas = np.concatenate([img1, img2], axis=1)
    fig, ax = plt.subplots(figsize=(16, 6))
    ax.imshow(canvas)
    rng = np.random.default_rng(42)
    for i1, i2 in matches[:max_display]:
        r1, c1 = kp1[i1]
        r2, c2 = kp2[i2]
        color = rng.random(3)
        ax.plot([c1, c2 + W1], [r1, r2], '-', color=color, linewidth=0.6, alpha=0.7)
        ax.plot(c1, r1, '.', color=color, markersize=4)
        ax.plot(c2 + W1, r2, '.', color=color, markersize=4)
    ax.set_title(f'{title} — {len(matches)} matches shown')
    ax.axis('off')
    plt.tight_layout()
    plt.show()


matches = match_descriptors_ssd(desc0, desc1)
print(f'{len(matches)} matches found')
visualize_matches(images[0], valid0, images[1], valid1, matches, title='SSD matches')

105 matches found


/var/folders/0h/51gcfhf96z5cb9mhps40vflh0000gn/T/ipykernel_23803/41376996.py:56: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


## 5. Homography estimation with RANSAC
> **New concept**: homography = projective transformation between two planes.  
> `cv2.findHomography` with RANSAC filters outlier matches automatically.

In [6]:
def estimate_homography(
    kp_src: np.ndarray,
    kp_dst: np.ndarray,
    match_indices: np.ndarray,
    ransac_reproj_threshold: float = 5.0,
) -> tuple[np.ndarray, np.ndarray]:
    """
    Estimate homography H such that dst_pt ≈ H @ src_pt.

    Parameters
    ----------
    kp_src : np.ndarray
        Source keypoints (N, 2) in (row, col).
    kp_dst : np.ndarray
        Destination keypoints (M, 2) in (row, col).
    match_indices : np.ndarray
        Matched pairs (K, 2) — (src_idx, dst_idx).
    ransac_reproj_threshold : float
        Maximum reprojection error for RANSAC inliers (pixels).

    Returns
    -------
    H : np.ndarray
        3×3 homography matrix.
    inlier_mask : np.ndarray
        Boolean mask of inliers among matches.
    """
    # cv2.findHomography expects (x, y) = (col, row)
    src_pts = kp_src[match_indices[:, 0]][:, ::-1].astype(np.float32)  # (col, row)
    dst_pts = kp_dst[match_indices[:, 1]][:, ::-1].astype(np.float32)

    H, mask = cv2.findHomography(
        src_pts, dst_pts,
        method=cv2.RANSAC,
        ransacReprojThreshold=ransac_reproj_threshold,
    )
    return H, mask.ravel().astype(bool)


H, inlier_mask = estimate_homography(valid0, valid1, matches)

print(f'Homography matrix H:')
print(np.round(H, 4))
print(f'\nInliers: {inlier_mask.sum()} / {len(matches)}')

# Visualize inlier matches only
visualize_matches(images[0], valid0, images[1], valid1,
                  matches[inlier_mask], title='Inlier matches after RANSAC')

Homography matrix H:
[[-1.3940000e-01  4.0500000e-02  4.8937282e+03]
 [-4.6780000e-01  8.9180000e-01  5.3275200e+01]
 [-2.0000000e-04 -0.0000000e+00  1.0000000e+00]]

Inliers: 41 / 105


/var/folders/0h/51gcfhf96z5cb9mhps40vflh0000gn/T/ipykernel_23803/41376996.py:56: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


## 6. Warp images to a common coordinate system
> **New concept**: `cv2.warpPerspective` applies the homography to map image 0 into image 1's reference frame.

In [7]:
def warp_image(
    src: np.ndarray,
    H: np.ndarray,
    output_shape: tuple[int, int],
) -> np.ndarray:
    """
    Warp src image using homography H into a canvas of given shape.

    Parameters
    ----------
    src : np.ndarray
        Source image (H, W, 3) in RGB.
    H : np.ndarray
        3×3 homography matrix.
    output_shape : tuple (height, width)
        Size of the output canvas.

    Returns
    -------
    np.ndarray
        Warped image on the output canvas.
    """
    h, w = output_shape
    warped = cv2.warpPerspective(src, H, (w, h),
                                 flags=cv2.INTER_LINEAR,
                                 borderMode=cv2.BORDER_CONSTANT,
                                 borderValue=0)
    return warped


# Compute canvas size large enough to hold both images
H_img, W_img = images[0].shape[:2]
canvas_h, canvas_w = H_img, W_img + images[1].shape[1]

warped0 = warp_image(images[0], H, (canvas_h, canvas_w))

fig, axes = plt.subplots(1, 2)
axes[0].imshow(warped0)
axes[0].set_title('Image 0 warped into image 1 frame')
axes[0].axis('off')
axes[1].imshow(images[1])
axes[1].set_title('Image 1 (reference)')
axes[1].axis('off')
plt.tight_layout()
plt.show()

/var/folders/0h/51gcfhf96z5cb9mhps40vflh0000gn/T/ipykernel_23803/2799663436.py:45: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


## 7. Blending
> **Course concept**: linear blending / alpha compositing in the overlap region.  
> Optional bonus: multi-band blending using Laplacian pyramid in the frequency domain.

In [8]:
def alpha_blend(
    img_warped: np.ndarray,
    img_ref: np.ndarray,
) -> np.ndarray:
    """
    Blend a warped image with a reference image on a common canvas.

    In the overlap zone, pixel values are averaged.
    In non-overlapping zones, the non-zero image is used.

    Parameters
    ----------
    img_warped : np.ndarray
        Warped image on the full canvas (H, W, 3), uint8.
    img_ref : np.ndarray
        Reference image placed on the canvas (H, W, 3), uint8.

    Returns
    -------
    np.ndarray
        Blended panorama (H, W, 3), uint8.
    """
    canvas_h, canvas_w = img_warped.shape[:2]

    # Masks: non-zero pixels in each image
    mask_warped = (img_warped.sum(axis=2) > 0).astype(np.float32)
    mask_ref    = (img_ref.sum(axis=2)    > 0).astype(np.float32)

    # Alpha = proportion from each image
    overlap = mask_warped * mask_ref
    alpha_w  = np.where(overlap, 0.5, mask_warped)[..., np.newaxis]
    alpha_r  = np.where(overlap, 0.5, mask_ref   )[..., np.newaxis]

    blended = (img_warped.astype(np.float32) * alpha_w
             + img_ref.astype(np.float32)    * alpha_r)
    return np.clip(blended, 0, 255).astype(np.uint8)


# Place image 1 on the canvas (no warp needed — it is the reference)
canvas_ref = np.zeros((canvas_h, canvas_w, 3), dtype=np.uint8)
canvas_ref[:images[1].shape[0], :images[1].shape[1]] = images[1]

panorama = alpha_blend(warped0, canvas_ref)

plt.figure(figsize=(18, 6))
plt.imshow(panorama)
plt.title('Panorama — alpha blending')
plt.axis('off')
plt.tight_layout()
plt.show()

/var/folders/0h/51gcfhf96z5cb9mhps40vflh0000gn/T/ipykernel_23803/305223326.py:50: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


## 7b. (Optional) Multi-image stitching
> Extend the pipeline to stitch N > 2 images sequentially.

In [9]:
def stitch_all(images: list[np.ndarray]) -> np.ndarray:
    """
    Stitch a list of overlapping images into a single panorama.

    The first image is used as the reference frame.
    Each subsequent image is warped and blended into the growing canvas.

    Parameters
    ----------
    images : list of np.ndarray
        Ordered list of RGB images with ~30-50% overlap between consecutive ones.

    Returns
    -------
    np.ndarray
        Final panorama image.
    """
    # TODO: implement sequential stitching
    # Hint: accumulate the canvas, warp each new image, blend
    raise NotImplementedError('Implement sequential stitching here')


# panorama_all = stitch_all(images)
# plt.imshow(panorama_all)
# plt.axis('off')
# plt.show()

## 8. Save result

In [10]:
out_path = OUTPUT_DIR / 'panorama_result.jpg'
cv2.imwrite(str(out_path), cv2.cvtColor(panorama, cv2.COLOR_RGB2BGR))
print(f'Panorama saved to {out_path}')

Panorama saved to ../data/output/panorama_result.jpg


## 9. Evaluation & discussion
- Visually inspect seams and ghosting in the overlap zone
- Count inlier ratio (step 5) — lower → worse match quality
- Try different images (more/less overlap, different lighting)
- Comment on what fails and why in your `report.pdf`

In [11]:
print('=== Pipeline summary ===')
print(f'Corners detected (img0):   {len(corners0)}')
print(f'Corners detected (img1):   {len(corners1)}')
print(f'Matches (after ratio test): {len(matches)}')
print(f'Inliers (after RANSAC):     {inlier_mask.sum()}')
print(f'Inlier ratio:               {inlier_mask.sum() / max(len(matches), 1):.2%}')
print(f'Output saved to:            {out_path}')

=== Pipeline summary ===
Corners detected (img0):   6696
Corners detected (img1):   14805
Matches (after ratio test): 105
Inliers (after RANSAC):     41
Inlier ratio:               39.05%
Output saved to:            ../data/output/panorama_result.jpg
